# NLP I — テキスト前処理と古典手法

テキストの前処理と、BoW / TF-IDF による特徴量化＋線形モデルの強いベースラインを学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/deep-learning/06-nlp-classic`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# 訓練文書と未見のテスト文書を分離して用意します（TF-IDF は訓練文書のみで fit）
train_text = [
    "Cross validation prevented overfitting and improved generalization.",
    "Feature engineering with domain insights boosted validation AUC.",
    "Blending diverse models produced a stable leaderboard result.",
    "Clean pipeline design eliminated target leakage across folds.",
    "Data leakage from global scaling ruined private leaderboard score.",
    "Unvalidated threshold tuning overfit badly to public test split.",
    "Corrupted labels and unaligned columns caused silent prediction errors.",
    "Ignoring group structure leaked user history into validation folds.",
]
y = [1, 1, 1, 1, 0, 0, 0, 0]

test_text = [
    "Out-of-fold validation gave reliable ensemble weights.",
    "Global imputation before splitting leaked validation statistics.",
]

## まず軽い手法でベースライン

NLP コンペでも、いきなり大型モデルではなく、**TF-IDF + 線形モデル**で
強いベースラインを作るのが定石です。速く、安定し、アンサンブルの素材にもなります。

## テキスト前処理

- 小文字化・記号や空白の正規化
- （言語により）トークン化。日本語は分かち書きが必要
- 不要語の除去はタスク次第（過度な除去は情報を落とす）

## BoW / TF-IDF

- **BoW（Bag of Words）**：単語の出現回数を特徴にする
- **TF-IDF**：頻出しすぎる語を抑え、その文書に特徴的な語を重視する

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X = tfidf.fit_transform(train_text)        # 訓練データだけで fit
X_test = tfidf.transform(test_text)        # テストは transform のみ

model = LogisticRegression(max_iter=1000).fit(X, y)

In [ ]:
test_probs = model.predict_proba(X_test)[:, 1]
for txt, p in zip(test_text, test_probs):
    print(f"prob={p:.3f} | {txt}")

> **Warning:** `fit` は**訓練データだけ**に行い、検証/テストには `transform` を適用します。全体で fit すると 語彙や IDF に検証側の情報が混ざりデータリークになります（交差検証の fold 内でも同様）。

> **Tip:** TF-IDF は大量の疎な特徴を生みます。ロジスティック回帰や線形 SVM、LightGBM が相性良く、 次レッスンの Transformer 系と**アンサンブル**すると多様性が出ます。

## 次のステップ

次は Transformer / BERT と Hugging Face によるファインチューニングを学びます。